# Opisy canonical values — słownik pojęć katalogu

Dla każdej canonical value z `zurada_canonical_mappings.json` generujemy krótki opis po polsku.  
Wynik: `zurada_canonical_descriptions.csv` z kolumnami `value`, `column`, `description`.

Przetwarzamy batchami (20 wartości / request) → ~18 wywołań API łącznie.

In [1]:
%pip install openai pandas tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, time
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "Brak klucza API")  # ustaw zmienną środowiskową lub wklej tu
MODEL          = 'gpt-5.5'
CANONICAL_JSON = 'zurada_canonical_mappings.json'
CHECKPOINT_CSV = 'zurada_descriptions_checkpoint.csv'
OUTPUT_CSV     = 'zurada_canonical_descriptions.csv'
BATCH_SIZE     = 20
SLEEP_BETWEEN  = 0.4
MAX_RETRIES    = 3

client = OpenAI(api_key=OPENAI_API_KEY)
print(f'Model: {MODEL}')


Model: gpt-5.5


In [3]:
with open(CANONICAL_JSON, encoding='utf-8') as f:
    mappings = json.load(f)

COL_CONTEXT = {
    'dozwolone_powierzchnie': 'powierzchnia lub materiał, na którym produkt czyszczący może być bezpiecznie stosowany',
    'odradzane_powierzchnie': 'powierzchnia lub materiał, na którym NIE należy stosować produktu — może go uszkodzić',
    'metoda_mycia':           'metoda lub technika aplikacji/użycia produktu czyszczącego',
    'zgodnosc_i_certyfikaty': 'certyfikat, norma lub standard jakości/bezpieczeństwa związany z produktem',
    'ostrzezenia_bhp':        'ostrzeżenie lub zalecenie bezpieczeństwa przy stosowaniu produktu',
    'odczyn_ph':              'poziom odczynu pH produktu czyszczącego',
}

records = []
seen = set()
for col, info in mappings.items():
    for value in info['canonical_values']:
        key = (value, col)
        if key not in seen:
            seen.add(key)
            records.append({'value': value, 'column': col})

print(f'Lacznie {len(records)} wpisow do opisania ({len(set(r["value"] for r in records))} unikalnych wartosci).')
pd.DataFrame(records).groupby('column').size().rename('count')


Lacznie 360 wpisow do opisania (351 unikalnych wartosci).


column
dozwolone_powierzchnie    201
metoda_mycia               23
odczyn_ph                   5
odradzane_powierzchnie     48
ostrzezenia_bhp            63
zgodnosc_i_certyfikaty     20
Name: count, dtype: int64

In [4]:
SYSTEM_PROMPT = (
    'Jestes ekspertem ds. chemii czyszczecej i higieny przemyslowej.\n'
    'Tworzysz slownik pojec dla systemu rekomendacji produktow czyszczacych.\n\n'
    'Dla kazdej podanej wartosci napisz KROTKI opis (1-2 zdania) po polsku.\n'
    'Opis powinien:\n'
    '- Wyjasniać CO TO JEST w kontekscie chemii czyszczecej / higieny\n'
    '- Byc zrozumialy dla osoby zamawiajacaej srodki czystosci (nie chemika)\n'
    '- Dla skrotow (HACCP, CIP, RSPO, IFRA, EU Ecolabel) — wyjasnic co oznacza skrot i dlaczego jest wazny\n'
    '- Dla powierzchni — opisac material i czemu wymaga uwagi przy czyszczeniu\n'
    '- Dla metod — opisac technikę aplikacji\n'
    '- Dla BHP — opisac na czym polega ryzyko lub zalecenie\n'
    '- NIE zaczynac od "To jest" — pisac wprost\n\n'
    'Odpowiadasz wylacznie JSON: {"descriptions": {"wartosc": "opis", ...}}'
)


def describe_batch(batch):
    lines = []
    for item in batch:
        ctx = COL_CONTEXT.get(item['column'], item['column'])
        lines.append(f'- "{item["value"]}" (kontekst: {ctx})')

    user_msg = (
        f'Opisz ponizsze {len(batch)} pojec z katalogu produktow czyszczacych:\n\n'
        + '\n'.join(lines)
        + '\n\nZwroc JSON: {"descriptions": {"wartosc": "opis", ...}}'
    )

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user',   'content': user_msg},
                ],
                response_format={'type': 'json_object'},
                max_completion_tokens=2000,
            )
            content = response.choices[0].message.content
            if not content:
                raise ValueError(f'Pusta odpowiedz, refusal={getattr(response.choices[0].message, "refusal", None)}')
            return json.loads(content).get('descriptions', {})
        except Exception as e:
            wait = 2 ** attempt
            print(f'  [RETRY {attempt}/{MAX_RETRIES}] — {e} — czekam {wait}s')
            time.sleep(wait)

    return {}


print('Funkcja describe_batch gotowa.')


Funkcja describe_batch gotowa.


In [5]:
if os.path.exists(CHECKPOINT_CSV):
    done_df   = pd.read_csv(CHECKPOINT_CSV)
    done_keys = set(zip(done_df['value'], done_df['column']))
    print(f'Checkpoint: {len(done_df)} opisow juz gotowych.')
else:
    done_df   = pd.DataFrame(columns=['value', 'column', 'description'])
    done_keys = set()
    print('Brak checkpointu — zaczynam od zera.')

todo = [r for r in records if (r['value'], r['column']) not in done_keys]
print(f'Do opisania: {len(todo)} wpisow w {-(-len(todo)//BATCH_SIZE)} batchach.')


Brak checkpointu — zaczynam od zera.
Do opisania: 360 wpisow w 18 batchach.


In [6]:
new_rows = []
batches  = [todo[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]

for batch in tqdm(batches, desc='Generowanie opisow'):
    first_val = batch[0]['value']
    print(f'  batch: {first_val!r} ... ({len(batch)} wpisow)')

    descriptions = describe_batch(batch)

    for item in batch:
        desc = descriptions.get(item['value'], '')
        new_rows.append({'value': item['value'], 'column': item['column'], 'description': desc})

    combined = pd.concat([done_df, pd.DataFrame(new_rows)], ignore_index=True)
    combined.to_csv(CHECKPOINT_CSV, index=False)

    time.sleep(SLEEP_BETWEEN)

print(f'Gotowe. Lacznie {len(done_df) + len(new_rows)} opisow.')


Generowanie opisow:   0%|          | 0/18 [00:00<?, ?it/s]

  batch: 'wszystkie powierzchnie' ... (20 wpisow)
  batch: 'powierzchnie kuchenne' ... (20 wpisow)
  batch: 'naczynia stalowe' ... (20 wpisow)
  batch: 'silniki' ... (20 wpisow)
  batch: 'powierzchnie matowe' ... (20 wpisow)
  batch: 'terakota' ... (20 wpisow)
  batch: 'parkiety' ... (20 wpisow)
  batch: 'balustrady' ... (20 wpisow)
  batch: 'bez kontaktu z żywnością' ... (20 wpisow)
  batch: 'opony' ... (20 wpisow)
  batch: 'porowate powierzchnie' ... (20 wpisow)
  batch: 'drewno' ... (20 wpisow)
  batch: 'armatura łazienkowa' ... (20 wpisow)
  batch: 'myjnie samochodowe' ... (20 wpisow)
  batch: 'bez dodecylobenzenosulfonianów' ... (20 wpisow)
  batch: 'spłukać naczynia wodą pitną' ... (20 wpisow)
  batch: 'instrukcja urządzenia' ... (20 wpisow)
  batch: 'aplikować na ciepło' ... (20 wpisow)
Gotowe. Lacznie 360 opisow.


In [7]:
final = pd.read_csv(CHECKPOINT_CSV)

empty_mask = final['description'].isna() | (final['description'].str.strip() == '')
print(f'Puste opisy: {empty_mask.sum()} wpisow')
final.loc[empty_mask, 'description'] = final.loc[empty_mask, 'value']

final = final.sort_values(['column', 'value']).reset_index(drop=True)
final.to_csv(OUTPUT_CSV, index=False)
print(f'Zapisano {len(final)} wpisow do {OUTPUT_CSV!r}')
final.head(10)


Puste opisy: 0 wpisow
Zapisano 360 wpisow do 'zurada_canonical_descriptions.csv'


,value,column,description
0,PVC,dozwolone_powierzchnie,PVC to tworzywo sztuczne stosowane m.in. w wyk...
1,akcesoria kuchenne,dozwolone_powierzchnie,"Drobne wyposażenie kuchni, np. deski, pojemnik..."
2,aluminium,dozwolone_powierzchnie,"Lekki metal stosowany w sprzęcie kuchennym, ok..."
3,armatura,dozwolone_powierzchnie,"Baterie, krany, słuchawki prysznicowe i inne e..."
4,balustrady,dozwolone_powierzchnie,"Elementy poręczy i barierek, najczęściej ze st..."
5,bemary,dozwolone_powierzchnie,Podgrzewacze gastronomiczne do utrzymywania te...
6,beton,dozwolone_powierzchnie,Porowaty materiał mineralny stosowany na posad...
7,bez kontaktu z żywnością,dozwolone_powierzchnie,"Powierzchnie, które nie mają bezpośredniej sty..."
8,blachy,dozwolone_powierzchnie,"Płaskie elementy kuchenne lub piekarnicze, czę..."
9,blaty,dozwolone_powierzchnie,"Powierzchnie robocze w kuchniach, laboratoriac..."


In [8]:
print('=== Certyfikaty i zgodnosci ===')
cert_df = final[final['column'] == 'zgodnosc_i_certyfikaty']
for _, row in cert_df.iterrows():
    print(f'\n[{row["value"]}]')
    print(f'  {row["description"]}')


=== Certyfikaty i zgodnosci ===

[HACCP]
  HACCP oznacza Hazard Analysis and Critical Control Points, czyli system analizy zagrożeń i kontroli punktów krytycznych w bezpieczeństwie żywności. Produkty przydatne w obszarach HACCP pomagają utrzymać higienę w kuchniach, zakładach spożywczych i miejscach kontaktu z żywnością.

[IFRA]
  IFRA oznacza International Fragrance Association, czyli organizację ustalającą standardy bezpiecznego stosowania kompozycji zapachowych. Informacja o zgodności IFRA jest ważna przy produktach perfumowanych, bo pomaga ograniczać ryzyko podrażnień i niepożądanych reakcji na zapachy.

[PN-EN 13036-4]
  Polska wersja europejskiej normy dotyczącej pomiaru właściwości przeciwpoślizgowych nawierzchni metodą wahadła. W kontekście czyszczenia pomaga ocenić, czy podłoga po myciu zachowuje bezpieczną przyczepność.

[PN-EN 1500:2013]
  Norma określająca wymagania i metodę badania skuteczności higienicznej dezynfekcji rąk przez wcieranie. Ważna przy wyborze preparatów do 

In [9]:
print('=== Metody mycia ===')
for _, row in final[final['column'] == 'metoda_mycia'].iterrows():
    print(f'\n[{row["value"]}]')
    print(f'  {row["description"]}')


=== Metody mycia ===

[CIP]
  CIP oznacza Cleaning In Place, czyli mycie instalacji bez ich demontażu, np. rurociągów, zbiorników i urządzeń w przemyśle spożywczym. Metoda jest ważna, bo zapewnia powtarzalną higienę, oszczędza czas i ogranicza ryzyko zanieczyszczeń.

[aplikacja gąbką]
  Ręczne nanoszenie preparatu przy pomocy gąbki, zwykle z delikatnym pocieraniem powierzchni. Metoda dobra do miejscowych zabrudzeń, ale wymaga ostrożności na powierzchniach podatnych na zarysowania.

[aplikacja rękawicą]
  Nanoszenie produktu specjalną rękawicą czyszczącą lub myjącą, często stosowane przy samochodach, meblach i powierzchniach delikatnych. Pozwala równomiernie rozprowadzić środek i ograniczyć ryzyko uszkodzeń przy właściwym doborze materiału rękawicy.

[aplikator]
  Narzędzie lub element opakowania służący do precyzyjnego nanoszenia środka, np. spryskiwacz, dozownik, pad lub gąbka. Ułatwia kontrolę ilości produktu i równomierne pokrycie czyszczonej powierzchni.

[automatyczne]
  Aplikacja

In [10]:
SEARCH = 'CIP'
mask = (
    final['value'].str.contains(SEARCH, case=False, na=False)
    | final['description'].str.contains(SEARCH, case=False, na=False)
)
final[mask][['value', 'column', 'description']]


,value,column,description
201,CIP,metoda_mycia,"CIP oznacza Cleaning In Place, czyli mycie ins..."
